# Bước 3 — Fine-tune AdaptiveNPR (ablation kiến trúc + augmentation nhẹ)

Bước 2 cho thấy **AdaptiveNPR + Aug mạnh** cân bằng lại Real/Fake tốt nhất (Real-Acc 76.9→86.7, GANGen 31→82%) nhưng augmentation quá mạnh (σ≤3.0, JPEG q≥30) làm tụt accuracy tổng (85→80). Bước 3 tinh chỉnh:

| Thí nghiệm | Augmentation | Mục đích | Trạng thái |
|---|---|---|---|
| **adaptive_mild** | Blur p=0.1 σ∈[0,1] + JPEG p=0.1 q∈[60,100] | Điểm cân bằng: giữ lợi Real-Acc, không phá accuracy bộ dễ | ✅ Xong (results/adaptive_mild_*) — mất cân bằng GANGen (64.7, lệch real), không đạt |
| **adaptive_only** | Tắt | Ablation — tách riêng đóng góp kiến trúc AdaptiveNPR khỏi augmentation, xem module tự nó có giữ được accuracy ngang baseline không | ▶ Chạy phiên này |

**Vì sao chạy adaptive_only bây giờ:** feedback của thầy (2026-09-22) là accuracy/metric trung bình toàn bộ vẫn thấp hơn baseline (80.44 vs 84.98, F1 77.98 vs 87.48, AUC 86.81 vs 89.50) — chỉ thắng riêng GANGen. Nghi vấn: chính augmentation mạnh mới là thứ kéo accuracy xuống, không phải bản thân module AdaptiveNPR. adaptive_only kiểm tra trực tiếp giả thuyết đó.

**Chuẩn bị:** Accelerator = GPU T4/P100 · Add Input `deepfake-benchmark-zips` · **Internet = On** · Save Version → **Save & Run All (Commit)** để chạy ngầm.
Thời gian: ~3-4h train + tải checkpoint về, đánh giá 5 benchmark ở máy local (Docker CPU) như đã làm với adaptive_mild.

In [ ]:
EXPERIMENTS = ['adaptive_only']   # Đã xong adaptive_mild phiên trước. Phiên này (quota reset): adaptive_only
EPOCHS = 8
RUN_EVAL = False                  # False = chỉ train + lưu checkpoint (tiết kiệm ~1h GPU); đánh giá làm ở máy local

!rm -rf Deepfake-Detect && git clone -q --depth 1 https://github.com/KimThanhTran/Deepfake-Detect.git
%cd Deepfake-Detect
!pip -q install scikit-learn tqdm 2>/dev/null
import torch; print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

import glob, os
hits = sum((glob.glob(f'/kaggle/input/{d}/ForenSynths/ForenSynths/train') for d in ('*', '*/*', '*/*/*')), [])
assert hits, 'Không tìm thấy ForenSynths/ForenSynths/train trong /kaggle/input'
DATA = os.path.dirname(os.path.dirname(os.path.dirname(hits[0])))
print('DATA =', DATA)

In [ ]:
# Định nghĩa thí nghiệm: (cờ augmentation). Cả hai đều bật --adaptive_npr.
AUG = {
    'adaptive_only': '--blur_prob 0.0 --jpg_prob 0.0',
    'adaptive_mild': '--blur_prob 0.1 --blur_sig 0.0,1.0 --jpg_prob 0.1 --jpg_method cv2,pil --jpg_qual 60,100',
}
runs = [e for e in EXPERIMENTS if e in AUG]
print('Sẽ chạy:', runs)

for name in runs:
    !python train.py --name {name} --adaptive_npr \
        --dataroot {DATA}/ForenSynths/ForenSynths --classes car,cat,chair,horse \
        --batch_size 32 --lr 0.0002 --niter {EPOCHS} --num_threads 4 \
        --loss_freq 2000 --skip_bench_eval {AUG[name]}

In [ ]:
# Thu checkpoint
import glob, shutil, os
os.makedirs('/kaggle/working/ckpt', exist_ok=True)
ckpts = {}
for name in runs:
    c = sorted(glob.glob(f'checkpoints/{name}*/model_epoch_last.pth'))[-1]
    dst = f'/kaggle/working/ckpt/{name}.pth'
    shutil.copy(c, dst); ckpts[name] = dst
    print(name, '->', dst)

In [ ]:
# Đánh giá trên 5 benchmark (protocol NPR: resize 256, không crop) — chỉ khi RUN_EVAL=True
import os
if not RUN_EVAL:
    print('RUN_EVAL=False — bỏ qua đánh giá trên Kaggle. Tải thư mục ckpt/ về, đánh giá ở máy local.')
else:
    sets = {
        'ForenSynths-test':    f'{DATA}/ForenSynths/ForenSynths/test',
        'GANGen-Detection':    f'{DATA}/GANGen-Detection/GANGen-Detection',
        'UniversalFakeDetect': f'{DATA}/UniversalFakeDetect/UniversalFakeDetect',
        'DiffusionForensics':  f'{DATA}/DiffusionForensics/DiffusionForensics',
        'Diffusion1kStep':     f'{DATA}/Diffusion1kStep/Diffusion1kStep',
    }
    for name, ck in ckpts.items():
        for ds, root in sets.items():
            if not os.path.isdir(root):
                print('[skip]', ds); continue
            !python tools/eval_report.py --model_path {ck} --dataroot {root} \
                --out_dir /kaggle/working/results/{name}_{ds} \
                --label "{name} - {ds}" --batch_size 64 --num_workers 4

In [ ]:
# Tổng hợp (chỉ khi có kết quả eval trên Kaggle)
import glob
if RUN_EVAL and glob.glob('/kaggle/working/results/*/metrics.csv'):
    !python tools/augment_metrics.py /kaggle/working/results
    import pandas as pd
    print(pd.read_csv('/kaggle/working/results/summary_all_runs.csv').to_string(index=False))
else:
    print('Không có eval trên Kaggle. Checkpoint đã lưu ở /kaggle/working/ckpt/ — tải về đưa cho bước tổng hợp local.')